# GFP fluorescence prediction with representation transfer

Base model: ESM-2 35M (`facebook/esm2_t12_35M_UR50D`), a masked model.
Second model: ProGen2-small (`hugohrban/progen2-small`), an autoregressive model, so a different family.

Both encoders are kept frozen. We only train small parts on top (a mapping network and
a ridge regression head).

What this notebook compares:
- ESM only (the main baseline)
- ProGen2 only
- ESM + ProGen2 concatenated
- transfer: map ProGen2 into ESM space, average with ESM, then predict

The split is grouped by mutated position and is used as given. We fit on train, pick
settings on validation, then refit on train+val and score once on test.

Before running: Runtime -> Change runtime type -> GPU (T4 is enough).

In [ ]:
!pip install -q transformers tokenizers scikit-learn scipy pandas
# if the progen2 cell below fails to load the model, uncomment and run this, then rerun:
# !pip install -q transformers==4.44.2

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Load the data

Upload `train.csv`, `validation.csv` and `test.csv`. They can be loose in the Colab
files pane or inside a `dataset/` folder, the loader checks both.

In [ ]:
# from google.colab import files
# files.upload()   # then pick the three csv files

import os

def load(name):
    for p in [name, "dataset/" + name]:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name + " not found, upload it first")

train = load("train.csv")
val = load("validation.csv")
test = load("test.csv")
print(len(train), len(val), len(test))

# quick look
print(train[["mutant", "DMS_score"]].head())
print("lengths:", set(train["mutated_sequence"].str.len()))
print("all single mutations:", (pd.concat([train, val, test])["mutation_count"] == 1).all())

## Frozen encoders

We turn each sequence into one vector by mean pooling the last hidden layer over the
residues. For ESM we drop the special start/end tokens before pooling. For ProGen2 we
put a "1" in front, which is how that model marks reading direction, and pool over the
rest.

In [ ]:
from transformers import AutoTokenizer, AutoModel

esm_tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm_model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(device).eval()

def esm_embed(seqs, batch_size=32):
    feats = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i+batch_size]
        enc = esm_tok(batch, return_tensors="pt", padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            h = esm_model(**enc).last_hidden_state
        # mask out the cls token and the eos token, then mean pool
        mask = enc["attention_mask"].clone()
        mask[:, 0] = 0
        lengths = enc["attention_mask"].sum(1)
        for j, L in enumerate(lengths):
            mask[j, L-1] = 0
        mask = mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1)
        feats.append(pooled.cpu().numpy())
    return np.concatenate(feats)

In [ ]:
from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer

pg_model = AutoModelForCausalLM.from_pretrained("hugohrban/progen2-small", trust_remote_code=True).to(device).eval()
pg_tok = Tokenizer.from_pretrained("hugohrban/progen2-small")

def progen_embed(seqs, batch_size=16):
    feats = []
    for i in range(0, len(seqs), batch_size):
        batch = ["1" + s for s in seqs[i:i+batch_size]]   # 1 marks N to C direction
        ids = [e.ids for e in pg_tok.encode_batch(batch)]
        input_ids = torch.tensor(ids).to(device)
        with torch.no_grad():
            h = pg_model(input_ids, output_hidden_states=True).hidden_states[-1]
        pooled = h[:, 1:, :].mean(1)   # skip the direction token
        feats.append(pooled.float().cpu().numpy())
    return np.concatenate(feats)

In [ ]:
# extract features once and cache them so reruns are fast
def get_feats(name, seqs, fn):
    path = name + ".npy"
    if os.path.exists(path):
        return np.load(path)
    f = fn(list(seqs))
    np.save(path, f)
    return f

esm_train = get_feats("esm_train", train["mutated_sequence"], esm_embed)
esm_val   = get_feats("esm_val",   val["mutated_sequence"],   esm_embed)
esm_test  = get_feats("esm_test",  test["mutated_sequence"],  esm_embed)

pg_train = get_feats("pg_train", train["mutated_sequence"], progen_embed)
pg_val   = get_feats("pg_val",   val["mutated_sequence"],   progen_embed)
pg_test  = get_feats("pg_test",  test["mutated_sequence"],  progen_embed)

y_train = train["DMS_score"].values
y_val   = val["DMS_score"].values
y_test  = test["DMS_score"].values

print("esm dim", esm_train.shape[1], "progen dim", pg_train.shape[1])

## Metrics and the regression head

Spearman correlation is the main metric, MSE is reported too. The head is ridge
regression, one alpha to pick on validation. The target is left in its original units
so MSE is in those units.

In [ ]:
def evaluate(pred, true):
    rho = spearmanr(pred, true).correlation
    mse = float(((pred - true) ** 2).mean())
    return rho, mse

def pick_alpha(Xtr, ytr, Xval, yval):
    best = None
    for a in [0.1, 1, 10, 100, 1000]:
        m = Ridge(alpha=a).fit(Xtr, ytr)
        rho, mse = evaluate(m.predict(Xval), yval)
        if best is None or rho > best[1]:
            best = (a, rho, mse)
    return best   # alpha, val spearman, val mse

## Baselines (fit on train, select on validation)

Features are standardized using training statistics only.

In [ ]:
esm_scaler = StandardScaler().fit(esm_train)
pg_scaler = StandardScaler().fit(pg_train)

esm_tr = esm_scaler.transform(esm_train); esm_va = esm_scaler.transform(esm_val)
pg_tr = pg_scaler.transform(pg_train); pg_va = pg_scaler.transform(pg_val)

print("ESM only    ", pick_alpha(esm_tr, y_train, esm_va, y_val))
print("ProGen2 only", pick_alpha(pg_tr, y_train, pg_va, y_val))

concat_tr = np.concatenate([esm_tr, pg_tr], 1)
concat_va = np.concatenate([esm_va, pg_va], 1)
print("Concat      ", pick_alpha(concat_tr, y_train, concat_va, y_val))

## Transfer method

This follows the SoupFold idea of learning a map between two models' representations.
We train a small MLP to predict the ESM vector from the ProGen2 vector, using MSE on the
training set. Then we average the real ESM vector with the mapped ProGen2 vector and fit
ridge on that. Averaging in ESM space is the "soup" step, the mapped ProGen2 acts like a
second view of the same sequence.

The R2 printed below just checks how well the map reproduces ESM features on validation,
it is not the fitness metric.

In [ ]:
class Mapper(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(din, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, dout),
        )
    def forward(self, x):
        return self.net(x)

def train_mapper(src, tgt, epochs=300):
    src = torch.tensor(src, dtype=torch.float32, device=device)
    tgt = torch.tensor(tgt, dtype=torch.float32, device=device)
    m = Mapper(src.shape[1], tgt.shape[1]).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=3e-4)
    lossf = nn.MSELoss()
    for e in range(epochs):
        opt.zero_grad()
        loss = lossf(m(src), tgt)
        loss.backward()
        opt.step()
    return m

def map_feats(m, src):
    m.eval()
    with torch.no_grad():
        return m(torch.tensor(src, dtype=torch.float32, device=device)).cpu().numpy()

mapper = train_mapper(pg_tr, esm_tr, epochs=300)

pred_va = map_feats(mapper, pg_va)
ss_res = ((esm_va - pred_va) ** 2).sum()
ss_tot = ((esm_va - esm_va.mean(0)) ** 2).sum()
print("mapper val R2:", 1 - ss_res / ss_tot)

comb_tr = (esm_tr + map_feats(mapper, pg_tr)) / 2
comb_va = (esm_va + pred_va) / 2
print("Transfer    ", pick_alpha(comb_tr, y_train, comb_va, y_val))

## Freeze settings and refit on train+val, then test

Read the alphas off the validation results above and set them here. Everything learned
(scaler, mapper, ridge) is refit from scratch on the 871 train+val examples. Test data
only passes through the frozen encoders and the fitted parts, its labels are used for
scoring only.

In [ ]:
# set these from the validation prints above
alpha_esm = 10
alpha_concat = 10
alpha_transfer = 10

esm_dev = np.concatenate([esm_train, esm_val]); pg_dev = np.concatenate([pg_train, pg_val])
y_dev = np.concatenate([y_train, y_val])

esm_scaler = StandardScaler().fit(esm_dev)
pg_scaler = StandardScaler().fit(pg_dev)
esm_d = esm_scaler.transform(esm_dev); esm_t = esm_scaler.transform(esm_test)
pg_d = pg_scaler.transform(pg_dev); pg_t = pg_scaler.transform(pg_test)

# esm only
esm_pred = Ridge(alpha=alpha_esm).fit(esm_d, y_dev).predict(esm_t)
print("ESM only test:", evaluate(esm_pred, y_test))

# concat
concat_pred = Ridge(alpha=alpha_concat).fit(
    np.concatenate([esm_d, pg_d], 1), y_dev).predict(np.concatenate([esm_t, pg_t], 1))
print("Concat   test:", evaluate(concat_pred, y_test))

# transfer
mapper = train_mapper(pg_d, esm_d, epochs=300)
comb_d = (esm_d + map_feats(mapper, pg_d)) / 2
comb_t = (esm_t + map_feats(mapper, pg_t)) / 2
transfer_pred = Ridge(alpha=alpha_transfer).fit(comb_d, y_dev).predict(comb_t)
print("Transfer test:", evaluate(transfer_pred, y_test))

In [ ]:
# save predictions for each final method
for name, pred in [("esm", esm_pred), ("concat", concat_pred), ("transfer", transfer_pred)]:
    pd.DataFrame({"variant_id": test["variant_id"], "predicted_fitness": pred}).to_csv(
        "pred_" + name + ".csv", index=False)
print("saved pred_esm.csv, pred_concat.csv, pred_transfer.csv")

## Notes for the report

Fill these in from the numbers this notebook prints:
- validation Spearman for ESM, ProGen2, concat, transfer
- test Spearman and MSE for ESM, concat, transfer (ProGen2-only can be added the same way)
- compare transfer against the ESM baseline, that is the test of whether the second model helped
- the mapper R2 says how much of ESM the ProGen2 features can explain

A negative or flat result is fine to report. The two models see almost identical sequences
here (one substitution apart), so the second model may add little beyond ESM.